# 第 18 课：NumPy 字节交换

整数和浮点数在内存中由字节组成。不同设备或文件格式可能使用不同的字节序（endianness）：小端把低位字节放在前面，大端把高位字节放在前面。本课用 `byteswap` 和字节序 dtype 处理这类数据。

In [1]:
import sys
import numpy as np

print('本机字节序：', sys.byteorder)
print('NumPy 版本：', np.__version__)

本机字节序： little
NumPy 版本： 2.5.3


大端模式：指数据的高字节保存在内存的低地址中，而数据的低字节保存在内存的高地址中，这样的存储模式有点儿类似于把数据当作字符串顺序处理：地址由小向大增加，而数据从高位往低位放；这和我们的阅读习惯一致。

小端模式：指数据的高字节保存在内存的高地址中，而数据的低字节保存在内存的低地址中，这种存储模式将地址的高低和数据位权有效地结合起来，高地址部分权值高，低地址部分权值低。

## 1. 查看数组的字节

`tobytes()` 返回数组的原始字节，`hex()` 可以把它显示成十六进制。`dtype.byteorder` 表示字节序：`<` 小端、`>` 大端、`=` 本机字节序、`|` 表示不适用（例如单字节类型）。

In [8]:
a = np.array([1, 256, 8755], dtype=np.int16)

print('数组：', a)
print('dtype：', a.dtype)
print('byteorder：', a.dtype.byteorder)
print('原始字节：', a.tobytes().hex(' '))

数组： [   1  256 8755]
dtype： int16
byteorder： =
原始字节： 01 00 00 01 33 22


## 2. `byteswap` 交换每个元素的字节

`a.byteswap()` 返回一个新数组，交换字节但不修改 dtype 标记。因此它通常用于底层字节转换；如果只交换字节而不改变解释方式，看到的数值可能会变化。`inplace=True` 会直接修改原数组的字节。

In [ ]:
swapped = a.byteswap()

print('交换字节后的数值：', swapped)
print('交换后的字节：', swapped.tobytes().hex(' '))
print('原数组仍然是：', a)
    
copy_a = a.copy()
copy_a.byteswap(inplace=True)
print('原地交换后：', copy_a)

交换字节后的数值： [  256     1 13090]
交换后的字节： 00 01 01 00 22 33
原数组仍然是： [   1  256 8755]
原地交换后： [  256     1 13090]


## 3. 转换字节序并保持数值含义

如果目标是把数组转换为另一种字节序，同时保持元素代表的数值不变，应交换字节并使用新的 dtype 解释它们：`a.byteswap().view(a.dtype.newbyteorder('>'))`。只调用 `byteswap` 不足以完成这种转换。

In [ ]:
little = np.array([1, 256, 8755], dtype='<i2')
big = little.byteswap().view(little.dtype.newbyteorder('>'))

print('小端 dtype：', little.dtype, '数值：', little)
print('大端 dtype：', big.dtype, '数值：', big) 
print('大端字节：', big.tobytes().hex(' '))
print('数值是否保持：', np.array_equal(little, big))

小端 dtype： int16 数值： [   1  256 8755]
大端 dtype： >i2 数值： [   1  256 8755]
大端字节： 00 01 01 00 22 33
数值是否保持： True


## 4. 与文件或网络数据交换

创建明确的大端数组可使用 `dtype='>i2'`。读取外部数据后，先确认文件规定的字节序，再转换为本机常用 dtype。`astype` 会按数值转换并生成新数组，适合得到本机整数数组。

In [5]:
network_data = np.array([1000, 2000, 3000], dtype='>i2')
native_data = network_data.astype(np.int16)

print('外部大端数据：', network_data)
print('dtype：', network_data.dtype)
print('转换为本机 int16：', native_data)
print('转换后的 dtype：', native_data.dtype)

外部大端数据： [1000 2000 3000]
dtype： >i2
转换为本机 int16： [1000 2000 3000]
转换后的 dtype： int16


## 5. 注意事项

- `byteswap` 改变的是字节排列，不是数学运算。
- `inplace=True` 会修改原数组，处理共享数据时要谨慎。
- `uint8` 每个元素只有一个字节，字节序没有实际作用。
- 读写二进制文件时必须遵守文件格式规定，不能仅凭本机字节序猜测。

In [6]:
one_byte = np.array([1, 2, 255], dtype=np.uint8)
print('uint8 的 dtype：', one_byte.dtype)
print('uint8 的 byteorder：', one_byte.dtype.byteorder)
print('交换后仍为：', one_byte.byteswap())

uint8 的 dtype： uint8
uint8 的 byteorder： |
交换后仍为： [  1   2 255]


## 本节练习

创建 `np.array([258, 1025, 4097], dtype=np.uint16)`：

1. 输出其 dtype 和原始十六进制字节；2. 转换为大端 dtype 并确认数值不变；3. 将大端数组用 `astype(np.uint16)` 转回本机整数；4. 比较转换前后的数组。

In [7]:
values = np.array([258, 1025, 4097], dtype=np.uint16)
big_values = values.byteswap().view(values.dtype.newbyteorder('>'))
roundtrip = big_values.astype(np.uint16)

print('原 dtype：', values.dtype)
print('原始字节：', values.tobytes().hex(' '))
print('大端 dtype：', big_values.dtype)
print('大端字节：', big_values.tobytes().hex(' '))
print('转回本机整数：', roundtrip)
print('是否与原数组相等：', np.array_equal(values, roundtrip))

原 dtype： uint16
原始字节： 02 01 01 04 01 10
大端 dtype： >u2
大端字节： 01 02 04 01 10 01
转回本机整数： [ 258 1025 4097]
是否与原数组相等： True


### 小结

`byteswap` 适合交换底层字节；结合 `dtype.newbyteorder` 才能明确告诉 NumPy 应如何解释这些字节。处理二进制文件时，始终把字节序当作数据格式的一部分。下一课可以继续学习 NumPy 的副本与视图。